In [1]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import xarray as xr
import numpy as np

from tqdm import tqdm

In [2]:
# ==========================================
# 1. ПОДГОТОВКА ДАТАСЕТА
# ==========================================
class ERA5HighResDataset(Dataset):
    def __init__(self, nc_path):
        print("Загрузка 15 ГБ NetCDF файла... (потребуется от 32 ГБ ОЗУ)")
        ds = xr.open_dataset(nc_path)
        
        surf_vars = [
            '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
            '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
            'total_column_water_vapour', 'total_cloud_cover'
        ]
        
        atm_vars = [
            'temperature', 'u_component_of_wind', 'v_component_of_wind',
            'geopotential', 'specific_humidity'
        ]
        
        print("Извлечение наземных переменных...")
        surf_arrays = [ds[v].values[:, np.newaxis, :, :] for v in surf_vars]
        
        print("Извлечение атмосферных переменных...")
        atm_arrays = [ds[v].values for v in atm_vars]
        
        # Закрываем исходный датасет для экономии памяти
        ds.close()
        del ds
        gc.collect()
        
        print("Склейка тензоров...")
        self.data = np.concatenate(surf_arrays + atm_arrays, axis=1)
        
        # Очищаем временные списки
        del surf_arrays
        del atm_arrays
        gc.collect()

        print("Заполнение NaN и нормализация...")
        self.data = np.nan_to_num(self.data, nan=0.0)
        
        self.mean = np.mean(self.data, axis=(0, 2, 3), keepdims=True)
        self.std = np.std(self.data, axis=(0, 2, 3), keepdims=True)
        self.std[self.std == 0] = 1.0 
        
        self.data = (self.data - self.mean) / self.std
        
        # Для AMP лучше всего подавать float32 (PyTorch сам приведет к fp16 на GPU)
        self.data = self.data.astype(np.float32)
        
        self.H_orig = self.data.shape[2]
        self.W_orig = self.data.shape[3]
        
        # Динамический паддинг (для 721x1440 высота станет 728)
        pad_H = (8 - self.H_orig % 8) % 8
        pad_W = (8 - self.W_orig % 8) % 8
        
        self.data = np.pad(self.data, ((0,0), (0,0), (0, pad_H), (0, pad_W)), mode='constant', constant_values=0)
        print(f"Форма данных для обучения: {self.data.shape}")

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx])

In [3]:
# ==========================================
# 2. АРХИТЕКТУРА VAE
# ==========================================
class ConvVAE(nn.Module):
    def __init__(self, in_channels=28, latent_channels=128):
        super(ConvVAE, self).__init__()
        
        # Энкодер: 28 -> 64 -> 128 -> 256
        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        self.fc_mu = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        self.fc_var = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        
        # Декодер: 128 -> 256 -> 128 -> 64 -> 28
        self.dec_input = nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1)
        
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1)
        
    def encode(self, x):
        h = F.leaky_relu(self.enc1(x))
        h = F.leaky_relu(self.enc2(h))
        h = F.leaky_relu(self.enc3(h))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.dec_input(z))
        h = F.relu(self.dec1(h))
        h = F.relu(self.dec2(h))
        return self.dec3(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon_x = self.decode(z)
        return recon_x, mu, log_var

In [4]:
# ==========================================
# 3. ФУНКЦИЯ ПОТЕРЬ И ЦИКЛ ОБУЧЕНИЯ
# ==========================================
def vae_loss_function(recon_x, x, mu, log_var, H_orig, W_orig):
    # Обрезаем черный паддинг
    valid_recon = recon_x[:, :, :H_orig, :W_orig]
    valid_x = x[:, :, :H_orig, :W_orig]
    
    # MSE Loss - используем sum и делим на размер батча
    recon_loss = F.mse_loss(valid_recon, valid_x, reduction='sum') / x.size(0)
    
    # KLD Loss 
    kld_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    
    beta = 0.05  # Вес для KLD, для огромных картинок его нужно держать маленьким
    return recon_loss + beta * kld_loss, recon_loss, kld_loss

In [5]:
def main():
    data_path = 'data/era5_highres_sample_2019.nc'
    
    # Экстремально малый батч для разрешения 1440x721
    batch_size = 2 
    epochs = 50
    learning_rate = 1e-4
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используемое устройство: {device}")
    if torch.cuda.is_available():
        print(f"Видеокарта: {torch.cuda.get_device_name(0)}")

    # pin_memory=True ускоряет перенос данных из ОЗУ в VRAM
    dataset = ERA5HighResDataset(data_path)
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        drop_last=False,
        pin_memory=True, 
        num_workers=0 # Для загрузки данных напрямую из RAM
    )
    
    H_orig, W_orig = dataset.H_orig, dataset.W_orig

    model = ConvVAE(in_channels=28, latent_channels=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # GradScaler необходим для смешанной точности (AMP)
    scaler = torch.amp.GradScaler('cuda')

    print("\nНачало обучения (с использованием Mixed Precision)...")
    model.train()
    
    for epoch in range(epochs):
        epoch_loss, epoch_recon, epoch_kld = 0, 0, 0
        
        # Оборачиваем dataloader в tqdm
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1:03d}/{epochs}")
        
        for data in progress_bar:
            data = data.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                recon_batch, mu, log_var = model(data)
                loss, recon_loss, kld_loss = vae_loss_function(
                    recon_batch, data, mu, log_var, H_orig, W_orig
                )
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # Считаем текущие лоссы для метрики (умножаем на размер батча)
            batch_loss = loss.item() * data.size(0)
            epoch_loss += batch_loss
            epoch_recon += recon_loss.item() * data.size(0)
            epoch_kld += kld_loss.item() * data.size(0)
            
            # Обновляем текст справа от прогресс-бара на каждом шаге
            progress_bar.set_postfix({'Batch Loss': f"{batch_loss:,.0f}"})
            
        num_samples = len(dataset) 
        pixels_per_sample = 28 * 721 * 1440
        
        # Делим суммарный лосс на количество пикселей, чтобы получить среднее
        mean_mse = (epoch_recon / num_samples) / pixels_per_sample
        
        print(f"Epoch {epoch+1:03d}/{epochs} | "
              f"Total Sum: {epoch_loss / num_samples:,.0f} | "
              f"Mean MSE per pixel: {mean_mse:.5f}")

    print("\nОбучение завершено!")
    os.makedirs('models', exist_ok=True)
    torch.save(model.state_dict(), 'models/era5_highres_vae_weights.pth')
    print("Веса модели сохранены.")

if __name__ == "__main__":
    main()

Используемое устройство: cuda
Видеокарта: NVIDIA GeForce RTX 5070
Загрузка 15 ГБ NetCDF файла... (потребуется от 32 ГБ ОЗУ)
Извлечение наземных переменных...
Извлечение атмосферных переменных...
Склейка тензоров...
Заполнение NaN и нормализация...
Форма данных для обучения: (128, 28, 728, 1440)

Начало обучения (с использованием Mixed Precision)...


Epoch 001/50: 100%|██████████| 64/64 [00:04<00:00, 12.98it/s, Batch Loss=40,271,984]


Epoch 001/50 | Total Sum: 27,101,547 | Mean MSE per pixel: 0.93050


Epoch 002/50: 100%|██████████| 64/64 [00:04<00:00, 14.91it/s, Batch Loss=34,874,668]


Epoch 002/50 | Total Sum: 18,051,235 | Mean MSE per pixel: 0.61862


...

Epoch 048/50 | Total Sum: 1,365,977 | Mean MSE per pixel: 0.04264


Epoch 049/50: 100%|██████████| 64/64 [00:04<00:00, 14.84it/s, Batch Loss=2,577,857]


Epoch 049/50 | Total Sum: 1,348,520 | Mean MSE per pixel: 0.04204


Epoch 050/50: 100%|██████████| 64/64 [00:04<00:00, 14.87it/s, Batch Loss=2,587,176]


Epoch 050/50 | Total Sum: 1,337,072 | Mean MSE per pixel: 0.04165

Обучение завершено!
Веса модели сохранены.


In [7]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. АРХИТЕКТУРА VAE
# ==========================================
class ConvVAE(nn.Module):
    def __init__(self, in_channels=28, latent_channels=128):
        super(ConvVAE, self).__init__()
        
        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        self.fc_mu = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        self.fc_var = nn.Conv2d(256, latent_channels, kernel_size=3, padding=1)
        
        self.dec_input = nn.Conv2d(latent_channels, 256, kernel_size=3, padding=1)
        
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1)
        
    def encode(self, x):
        h = F.leaky_relu(self.enc1(x))
        h = F.leaky_relu(self.enc2(h))
        h = F.leaky_relu(self.enc3(h))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.dec_input(z))
        h = F.relu(self.dec1(h))
        h = F.relu(self.dec2(h))
        return self.dec3(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon_x = self.decode(z)
        return recon_x, mu, log_var


# ==========================================
# 2. ФУНКЦИЯ ОЦЕНКИ
# ==========================================
def evaluate_on_train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Устройство: {device}")

    train_file = 'data/era5_highres_sample_2019.nc'
    if not os.path.exists(train_file):
        raise FileNotFoundError(f"Файл {train_file} не найден!")

    print("Загрузка обучающего датасета в память...")
    ds = xr.open_dataset(train_file)
    
    # 1. Подготовка широты для весов (Latitude-weighted)
    latitudes = ds.latitude.values
    cos_lat = np.clip(np.cos(np.deg2rad(latitudes)), a_min=0, a_max=None)
    cos_lat_tensor = torch.tensor(cos_lat, dtype=torch.float32, device=device).view(1, 1, -1, 1)

    surface_vars = [
        '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
        '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
        'total_column_water_vapour', 'total_cloud_cover'
    ]
    atm_vars = ['temperature', 'u_component_of_wind', 'v_component_of_wind', 'geopotential', 'specific_humidity']

    # 2. Сборка тензоров (идентично процессу обучения)
    surf_arrays = [ds[v].values[:, np.newaxis, :, :] for v in surface_vars]
    atm_arrays = [ds[v].values for v in atm_vars]
    
    ds.close()
    del ds
    gc.collect()
    
    print("Склейка и нормализация данных...")
    data = np.concatenate(surf_arrays + atm_arrays, axis=1)
    del surf_arrays, atm_arrays
    gc.collect()

    data = np.nan_to_num(data, nan=0.0)
    
    # Считаем точные mean и std по всей выборке (это и есть sigma_train)
    data_mean = np.mean(data, axis=(0, 2, 3), keepdims=True)
    data_std = np.std(data, axis=(0, 2, 3), keepdims=True)
    data_std[data_std == 0] = 1.0 
    
    # Нормализуем данные
    data_normalized = (data - data_mean) / data_std
    data_normalized = data_normalized.astype(np.float32)

    H_orig, W_orig = data_normalized.shape[2], data_normalized.shape[3]
    pad_H = (8 - H_orig % 8) % 8
    pad_W = (8 - W_orig % 8) % 8
    data_padded = np.pad(data_normalized, ((0,0), (0,0), (0, pad_H), (0, pad_W)), mode='constant')

    # 3. Инициализация и загрузка модели
    model = ConvVAE(in_channels=28, latent_channels=128).to(device)
    try:
        model.load_state_dict(torch.load('models/era5_highres_vae_weights.pth', map_location=device))
        print("Веса модели успешно загружены.")
    except FileNotFoundError:
        print("ВНИМАНИЕ: Файл весов не найден. Модель будет выдавать случайные значения.")
    model.eval()

    # 4. Переменные для метрик
    num_channels = 28
    weighted_squared_errors = torch.zeros(num_channels, device=device)
    sum_of_weights = 0.0
    batch_size = 2
    num_samples = data_padded.shape[0]

    # Тензоры mean и std на GPU для денормализации
    mean_tensor = torch.tensor(data_mean, device=device)
    std_tensor = torch.tensor(data_std, device=device)

    print("\nНачало оценки батчей...")
    with torch.no_grad():
        for i in tqdm(range(0, num_samples, batch_size), desc="Оценка на Train"):
            x_batch = torch.tensor(data_padded[i : i+batch_size], device=device)
            
            # Оригинальные физические данные для этого батча (без паддинга)
            x_physical = torch.tensor(data[i : i+batch_size], device=device)

            with torch.amp.autocast('cuda'):
                recon_x, _, _ = model(x_batch)
            
            # Обрезаем паддинг и денормализуем
            recon_x_cropped = recon_x[:, :, :H_orig, :W_orig]
            recon_x_physical = recon_x_cropped * std_tensor + mean_tensor
            
            # Latitude-weighted MSE
            squared_error = (recon_x_physical - x_physical) ** 2
            weighted_se = squared_error * cos_lat_tensor
            
            weighted_squared_errors += weighted_se.sum(dim=(0, 2, 3))
            sum_of_weights += cos_lat_tensor.sum() * W_orig * x_batch.size(0)

    # 5. Итоговые расчеты
    print("\nРасчет NRMSE...")
    rmse_per_channel = torch.sqrt(weighted_squared_errors / sum_of_weights)
    
    # sigma_train — это вектор стандартных отклонений, который мы рассчитали выше (размерность: 28)
    sigma_f_train = std_tensor.squeeze() 
    
    nrmse_per_channel = rmse_per_channel / sigma_f_train
    
    nrmse_surface = nrmse_per_channel[:8]
    nrmse_pressure = nrmse_per_channel[8:]
    
    score_surface = nrmse_surface.mean().item()
    score_pressure = nrmse_pressure.mean().item()
    score_all = 0.5 * score_surface + 0.5 * score_pressure
    
    print("\n" + "="*45)
    print(" РЕЗУЛЬТАТЫ ОЦЕНКИ НА ОБУЧАЮЩИХ ДАННЫХ (2019)")
    print("="*45)
    print(f"Surface Score (S_surface) : {score_surface:.5f}")
    print(f"Pressure Score (S_pressure): {score_pressure:.5f}")
    print(f"Overall Score (S_all)     : {score_all:.5f}")
    print("-" * 45)
    
    print("Детализация NRMSE по наземным полям (Surface):")
    for i in range(8):
        print(f"  {surface_vars[i]:<25}: {nrmse_surface[i].item():.5f}")

if __name__ == "__main__":
    evaluate_on_train()

Устройство: cuda
Загрузка обучающего датасета в память...
Склейка и нормализация данных...
Веса модели успешно загружены.

Начало оценки батчей...


Оценка на Train: 100%|██████████| 64/64 [00:03<00:00, 18.45it/s]



Расчет NRMSE...

 РЕЗУЛЬТАТЫ ОЦЕНКИ НА ОБУЧАЮЩИХ ДАННЫХ (2019)
Surface Score (S_surface) : 0.24295
Pressure Score (S_pressure): 0.17136
Overall Score (S_all)     : 0.20716
---------------------------------------------
Детализация NRMSE по наземным полям (Surface):
  2m_temperature           : 0.15827
  mean_sea_level_pressure  : 0.08914
  10m_u_component_of_wind  : 0.18958
  10m_v_component_of_wind  : 0.20658
  total_precipitation_6hr  : 0.49500
  sea_surface_temperature  : 0.29168
  total_column_water_vapour: 0.17492
  total_cloud_cover        : 0.33846
